# Rental Marketplace – Recommendation + Price Suggestion
Synthetic data generation → EDA → training → evaluation for **two ML models**.

| Model | Input dataset | Target | Algorithm |
|---|---|---|---|
| Recommendation | impressions (`shown/clicked/booked/completed`, distance, price …) | `P(click)·P(book|click)` | 2× HistGradientBoosting (funnel) |
| Price suggestion | bookings (asking / agreed price, comparable median, demand …) | `log(agreed / comparable_median)` | 3× HistGB quantile (p10/p50/p90) |

Run from the `notebooks/` folder (the cell below fixes the import path).

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from src.synth_recommendation import generate_recommendation_data
from src.synth_pricing import generate_pricing_data
from src import train_recommendation as rec, train_pricing as pri

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

Path("../data").mkdir(exist_ok=True)
Path("../models").mkdir(exist_ok=True)

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

## 1. Recommendation dataset – synthesis

In [ ]:
rec_df = generate_recommendation_data(n_rows=300_000, seed=42)
rec_df.to_csv("../data/recommendation.csv", index=False)
print(rec_df.shape)
rec_df.head()

In [ ]:
# Funnel sanity-check + EDA
print("CTR            :", round(rec_df.clicked.mean(), 4))
print("book / click   :", round(rec_df.booked.sum() / rec_df.clicked.sum(), 4))
print("complete / book:", round(rec_df.completed.sum() / rec_df.booked.sum(), 4))
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
rec_df.groupby("category").clicked.mean().sort_values().plot.barh(ax=ax[0], title="CTR by category")
rec_df.groupby(pd.cut(rec_df.distance_km, [0, 1, 2, 5, 10, 20, 50, 200])).clicked.mean().plot(ax=ax[1], marker="o", title="CTR vs distance (km)")
rec_df.groupby("month").clicked.mean().plot(ax=ax[2], marker="o", title="CTR by month")
plt.tight_layout(); plt.show()

## 2. Recommendation – training (time-based split: train < Oct-2025, test ≥ Oct-2025)

In [ ]:
rec_bundle, rec_test = rec.train(rec_df)
pd.Series(rec_bundle["metrics"]).round(4)

In [ ]:
m = rec_bundle["metrics"]
pd.Series({"Random": m["ndcg@10_random_baseline"], "Popularity": m["ndcg@10_popularity_baseline"], "Our model": m["ndcg@10_model"]}).plot.bar(title="NDCG@10 (higher is better)", color=["grey","orange","green"])
plt.show()

In [ ]:
# Which features matter? (permutation importance on a sample of the test set)
from sklearn.inspection import permutation_importance
from src.features import build_rec_features
Xte = build_rec_features(rec_test, rec_bundle["stats"]).sample(15000, random_state=0)
yte = rec_test.loc[Xte.index, "clicked"]
pi = permutation_importance(rec_bundle["model_click"], Xte, yte, scoring="roc_auc", n_repeats=3, random_state=0)
pd.Series(pi.importances_mean, index=Xte.columns).sort_values().plot.barh(title="Permutation importance (click AUC drop)")
plt.show()

In [ ]:
# Recommend top-10 for one user
user = rec_test.user_id.iloc[0]
cand = rec_test[rec_test.user_id == user].drop_duplicates("item_id")[["user_id","item_id","distance_km","price","is_available","category","area","month","clicked","booked"]]
rec.recommend_top_k(rec_bundle, cand.drop(columns=["clicked","booked"]), k=10)[["item_id","category","area","distance_km","price","is_available","p_click","p_book_given_click","score"]]

In [ ]:
joblib.dump(rec_bundle, "../models/recommendation_model.joblib")

## 3. Pricing dataset – synthesis

In [ ]:
price_df = generate_pricing_data(n_rows=60_000, seed=7)
price_df.to_csv("../data/pricing.csv", index=False)
print(price_df.shape, "| booked rows:", price_df.actual_agreed_price_per_day.notna().sum())
price_df.head()

In [ ]:
b = price_df.dropna(subset=["actual_agreed_price_per_day"])
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
(b.actual_agreed_price_per_day / b.asking_price_per_day).hist(bins=50, ax=ax[0]); ax[0].set_title("agreed / asking (owners overprice)")
b.groupby("condition").actual_agreed_price_per_day.mean().plot.bar(ax=ax[1], title="Mean agreed price by condition")
b.plot.scatter("category_area_demand_30d", "actual_agreed_price_per_day", s=2, alpha=.3, ax=ax[2], title="Demand vs price")
plt.tight_layout(); plt.show()

## 4. Pricing – training (median + p10 + p90 quantile models)

In [ ]:
price_bundle, price_test = pri.train(price_df)
pd.Series(price_bundle["metrics"]).round(4)

In [ ]:
m = price_bundle["metrics"]
pd.Series({"Asking price": m["baseline_asking_price_mape"], "Comparable median": m["baseline_comparable_median_mape"], "Our model": m["mape"]}).plot.bar(title="MAPE vs agreed price (lower is better)", color=["grey","orange","green"])
plt.show()
s = price_test.sample(300, random_state=1).sort_values("actual_agreed_price_per_day")
plt.figure(figsize=(10,4)); x = range(len(s))
plt.fill_between(x, s.pred_low, s.pred_high, alpha=.25, label="p10–p90 range")
plt.plot(x, s.pred, label="suggested"); plt.scatter(x, s.actual_agreed_price_per_day, s=6, c="k", label="actual agreed"); plt.legend(); plt.show()

In [ ]:
# Suggest a price for brand-new listings
new_items = price_test.head(5).drop(columns=["actual_agreed_price_per_day","pred","pred_low","pred_high"])
pri.suggest_price(price_bundle, new_items)[["category","brand","condition","area","asking_price_per_day","comparable_median_price","suggested_price","price_low","price_high"]]

In [ ]:
joblib.dump(price_bundle, "../models/pricing_model.joblib")

## Next steps
* Replace the synthetic CSVs with real data (same column names) – nothing else changes.
* Use the **`mlops_monitoring`** project to serve, monitor drift/performance and retrain these models.